# 04 — Shot Noise & Your First Circuit

Close the qubit arc: sample the Born rule the way a real device does, watch the estimate converge, and run it all as a genuine Qiskit circuit.

**Prerequisites:** `01/01_amplitudes_and_superposition`, `01/03_the_bloch_sphere`, `01/04_measurement_and_born_rule`.

**What you'll be able to do**
- Explain shot noise, and why more shots converge to the Born probabilities.
- Build and run a one-qubit circuit in Qiskit and read its counts.
- Say precisely why a real, noisy device pushes us toward density matrices next.

You have done the groundwork: a qubit is two amplitudes (`01/01`), a point on the Bloch sphere (`01/03`), and a set of measurement probabilities (`01/04`). This notebook brings the three together and connects them to a real machine. Take a moment — that is a complete, working understanding of a single qubit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.quantum.states import KET_PLUS, born_probabilities, sample_counts

np.random.seed(42)
viz.use_course_style()

## 1. One outcome at a time: shot noise

A single measurement gives one bare $0$ or $1$ (notebook 03). To *estimate* the Born probabilities, a real device repeats the measurement many times — each repeat is a **shot** — and counts the outcomes. With few shots the estimate is noisy; with many it settles down. Let's measure $|+\rangle$ at 20, 200 and 2000 shots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, shots in zip(axes, [20, 200, 2000]):
    counts = sample_counts(KET_PLUS, shots=shots, seed=0)
    viz.plot_counts(counts, ax=ax)
    ax.set_title(f"{shots} shots", pad=10)
fig.suptitle("Measuring |+>: more shots converge toward the 50/50 Born probabilities",
             fontsize=15, fontweight="bold")
fig.tight_layout()
plt.show()

**Read the figure.** At 20 shots the split can look visibly lopsided (pure chance); by 2000 shots it is close to the true $50/50$. This fluctuation is **shot noise** — not a bug, but what sampling a random process looks like. The spread shrinks like $1/\sqrt{\text{shots}}$, which is why precise estimates cost many repetitions.

## 2. A real circuit with Qiskit

So far we sampled from the state vector directly. Let's instead *build* $|+\rangle$ the way a quantum computer does — by applying a **Hadamard gate** `H` to $|0\rangle$ — then run the circuit on a simulator.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc = QuantumCircuit(1, 1)
qc.h(0)            # |0> -> |+>
qc.measure(0, 0)   # measure in the computational basis
qc.draw("mpl")

In [ ]:
sim = AerSimulator()
result = sim.run(transpile(qc, sim), shots=2048, seed_simulator=42).result()
counts = result.get_counts()
print("counts:", counts)

viz.plot_counts(counts)
plt.show()

**Read the figure.** The simulator reproduces the ideal coin flip (close to $1024 / 1024$).

On **real hardware** the split is *not* exactly even — read-out and gate errors bias it. That is the honest reason the next notebooks need **density matrices**: a prepared state is never perfectly pure. The verified idiom to run this on an IBM QPU (Open Plan, free) is:

```python
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
service = QiskitRuntimeService()                         # uses your saved account (no token in code)
backend = service.least_busy(operational=True, simulator=False)
sampler = SamplerV2(mode=backend)
job = sampler.run([transpile(qc, backend)])
counts = job.result()[0].data.c.get_counts()
```

## 3. Dictionary update

The course grows one classical-to-quantum correspondence at a time. The qubit arc adds the Born rule — how a quantum state yields a classical probability.

| Classical | Quantum |
|-----------|---------|
| probability vector `p` | density matrix $\rho$ (diagonal $\Rightarrow$ classical) |
| marginal | partial trace |
| event probability $p(x)$ | **Born rule** $|\langle x|\psi\rangle|^2$ |

## Your turn

1. **Watch the noise shrink.** Re-run the shot-noise figure with `shots = [5, 50, 500, 5000]`. By eye, how fast does the spread close on $50/50$? Does it match the $1/\sqrt{\text{shots}}$ rule?
2. **A biased circuit.** Replace `qc.h(0)` with `qc.ry(theta, 0)` for a $\theta$ of your choosing. Predict $P(0)$ from notebook 03 ($\cos^2(\theta/2)$), then compare with the simulator counts.
3. **Build $|-\rangle$.** Apply `H` then `Z` to $|0\rangle$ and measure. What counts do you get — and why can a $Z$ measurement still not tell $|-\rangle$ from $|+\rangle$? (Callback to notebook 03's phase-blindness.)

## What you built

- You saw **shot noise**: sampling the Born rule converges to the true probabilities as shots grow.
- You built and ran a real one-qubit Qiskit circuit and read its measurement counts.
- You can now name why honest, noisy hardware forces a richer description than a state vector.

That closes the **qubit arc**: from two amplitudes, to a point on a sphere, to probabilities, to a running circuit. You have a full picture of one qubit — genuinely well done.

## What's next

A real prepared state is never perfectly pure, and a state vector cannot describe "a mixture of possible states". In `01/12_from_states_to_density_matrix` we meet the object that can — the density matrix $\rho$ — the central character of the rest of the course.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 1–2, Cambridge University Press (2010).
- Qiskit textbook, "Representing Qubit States".

**Previous:** `notebooks/01_QuantumFoundations/10_expectation_values.ipynb` · **Next:** `notebooks/01_QuantumFoundations/12_from_states_to_density_matrix.ipynb`